In [ ]:
# ============================================================
# ViT-SE + Modified Firefly Algorithm — Faithful Implementation
# Based on: "A modified vision transformer framework for
# image-based land cover segmentation in rural architectural
# design and planning" (Wassan et al., 2025)
#
# FAITHFULNESS NOTES:
#   - Equations 1–17 implemented exactly as written in paper
#   - Eq.11 : inertia factor ω (linear decay)
#   - Eq.12 : geometric factor ψ (tan-sigmoid, paper's own definition)
#   - Eq.13 : MFA movement (exponential attractiveness, rand()-0.5)
#   - SE block: global-average-pool → FC → sigmoid (Eq.5)
#   - CLS-token classification head (Eq.6)
#   - AdamW lr=0.0001 (Table 4), 80/20 split (Table 4)
#
# DIMENSION-ONLY ADJUSTMENTS (Colab T4 free, ~15 GB VRAM):
#   - Input 384×384 → 64×64 (EuroSAT native resolution)
#   - Patch 16×16 → 8×8   (keeps ≥64 patches per image)
#   - embed_dim 1024 → 256, depth 24 → 6, heads 16 → 4
#   - FFN dim 4096 → 1024  (same mlp_ratio=4 as paper)
#   - MFA swarm 50 → 10/20, maxIter 700 → 5/15 (time budget)
#   - Epochs 100 → 30/100 (QUICK/FULL switch below)
# ============================================================

# ── Install (uncomment if needed) ──────────────────────────
# !pip install optuna --quiet

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports & Global Config                       ║
# ╚══════════════════════════════════════════════════════════╝

import os, math, time, random, warnings, json
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import EuroSAT
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed()

# ── Device ─────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {torch.cuda.get_device_name(0)}  ({vram:.1f} GB)")

# ── Paths ──────────────────────────────────────────────────
DATA_DIR   = "/content/data"
OUTPUT_DIR = "/content/output"
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Dataset constants ──────────────────────────────────────
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation",
    "Highway", "Industrial", "Pasture",
    "PermanentCrop", "Residential", "River", "SeaLake"
]
NUM_CLASSES = 10
IMAGE_SIZE  = 64     # EuroSAT native — cannot change
PATCH_SIZE  = 8      # 64/8 = 8×8 = 64 patches (scaled from paper's 16×16)

# Paper Table 4 normalisation (ImageNet, common for RGB EuroSAT)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Batch size (Colab T4 safe) ─────────────────────────────
def get_batch_size():
    if not torch.cuda.is_available(): return 64
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 14: return 128
    if vram >= 10: return 96
    return 64
SAFE_BATCH = get_batch_size()
print(f"Batch   : {SAFE_BATCH}")

# ── Run mode ───────────────────────────────────────────────
# QUICK : faster, for testing  (~1-2 hrs on Colab free)
# FULL  : paper-scale epochs   (~5-6 hrs on Colab free)
RUN_MODE = "QUICK"

if RUN_MODE == "QUICK":
    FINAL_EPOCHS  = 30      # paper uses 100; scaled for Colab
    MFA_SWARM     = 10      # paper uses 50
    MFA_MAXITER   = 5       # paper uses 700
    QUICK_EPOCHS  = 5       # epochs for fitness evaluation
    OPTUNA_TRIALS = 10
else:
    FINAL_EPOCHS  = 100     # paper's value
    MFA_SWARM     = 20
    MFA_MAXITER   = 15
    QUICK_EPOCHS  = 8
    OPTUNA_TRIALS = 20

TUNING_METHODS = ["fixed", "mfa", "optuna"]

print(f"Mode    : {RUN_MODE}  "
      f"(final_epochs={FINAL_EPOCHS}, mfa_swarm={MFA_SWARM}, "
      f"mfa_iters={MFA_MAXITER})")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Dataset (paper Section: Details of dataset)   ║
# ╚══════════════════════════════════════════════════════════╝
# Paper: "80% training, 20% evaluation"
# Paper: "flipping, rotation, contrast modifications"
# Paper: normalisation of pixel intensities

def get_transforms() -> Tuple[transforms.Compose, transforms.Compose]:
    """
    Paper states: flip, rotation, contrast modifications, normalisation.
    Val transform: resize + normalise only (no augmentation at test time).
    """
    train_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),          # paper: flipping
        transforms.RandomVerticalFlip(p=0.5),            # paper: flipping
        transforms.RandomRotation(degrees=15),            # paper: rotation
        transforms.ColorJitter(brightness=0.2,
                               contrast=0.2),             # paper: contrast mods
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),# paper: normalisation
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, val_tf


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        return (self.transform(img) if self.transform else img), label


_CACHE = None
def get_base_dataset():
    global _CACHE
    if _CACHE is None:
        from tqdm import tqdm
        raw = EuroSAT(root=DATA_DIR, download=True, transform=None)
        print("Caching EuroSAT to RAM...")
        _CACHE = [raw[i] for i in tqdm(range(len(raw)), ncols=70)]
        print(f"Cached {len(_CACHE):,} images.")

    class _ListDS(torch.utils.data.Dataset):
        def __init__(self, d): self.d = d
        def __len__(self): return len(self.d)
        def __getitem__(self, i): return self.d[i]
    return _ListDS(_CACHE)


def get_dataloaders(batch_size: int = SAFE_BATCH,
                    train_ratio: float = 0.8
                    ) -> Tuple[DataLoader, DataLoader]:
    """Paper: 80/20 split."""
    train_tf, val_tf = get_transforms()
    base = get_base_dataset()
    n    = len(base)
    idx  = list(range(n)); random.seed(SEED); random.shuffle(idx)
    split     = int(n * train_ratio)
    train_ds  = TransformSubset(base, idx[:split], train_tf)
    val_ds    = TransformSubset(base, idx[split:], val_tf)
    kw = dict(num_workers=2, pin_memory=True)
    print(f"  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw),
            DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw))


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — ViT-SE Model (paper Equations 1–6)           ║
# ╚══════════════════════════════════════════════════════════╝

class SEBlock(nn.Module):
    """
    Paper Eq. 5 — Squeeze-and-Excitation block.
    gp  = (1/H·W) Σ T_{a,b,s}          ← global average pool (spatial squeeze)
    f1  = σ(w2 · φ(w1 · s))            ← two FC layers, ReLU then Sigmoid
    out = T_{a,b,p} · fc               ← channel-wise recalibration

    In ViT context: x has shape (B, N, C).
    We treat N tokens as the "spatial" dimension → pool over dim=1.
    This matches the paper's intent of squeezing spatial info into a channel
    descriptor.
    """
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.w1 = nn.Linear(channels, hidden)   # w1 in paper
        self.w2 = nn.Linear(hidden, channels)   # w2 in paper

    def forward(self, x):
        # gp: global average pool over token/spatial dimension (B,N,C) → (B,C)
        s  = x.mean(dim=1)                              # Eq.5 : gp
        s  = F.relu(self.w1(s))                         # Eq.5 : φ(w1·s) — ReLU
        fc = torch.sigmoid(self.w2(s))                  # Eq.5 : σ(w2·...)
        return x * fc.unsqueeze(1)                      # Eq.5 : T·fc


class MultiHeadSelfAttention(nn.Module):
    """
    Paper Eq. 2 — Multi-headed self-attention.
    φ_A(α,β,δ) = Softmax(αβᵀ/√d_h) · δ
    φ_MHSA(T)  = ⊕(h1,h2,...) · W0
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5          # 1/√d_h in Eq.2
        self.qkv  = nn.Linear(embed_dim, embed_dim * 3)
        self.W0   = nn.Linear(embed_dim, embed_dim)     # W0 in Eq.2
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        # Eq.2 : Softmax(q·kᵀ / √d_h) · v
        attn = self.drop(
            F.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1))
        out  = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.W0(out)                             # Eq.2 : · W0


class TransformerSEBlock(nn.Module):
    """
    One Transformer encoder block + SE block (Fig. 3 of paper).

    Flow per block:
      x → MHSA (Eq.2) → residual + LayerNorm (Eq.3) → T'
      T' → FFN  (Eq.4) → residual + LayerNorm        → T''
      T'' → SE  (Eq.5) → output
    """
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0,
                 dropout=0.1, se_reduction=16):
        super().__init__()
        mlp_hidden = int(embed_dim * mlp_ratio)         # paper: dim_ff = 4×embed

        # Eq.3 components
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.drop1 = nn.Dropout(dropout)

        # Eq.4 components — paper: two linear layers with GeLU (φ)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),                                   # Eq.4: φ = GeLU
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

        # Eq.5 — SE block after each transformer block
        self.se = SEBlock(embed_dim, reduction=se_reduction)

    def forward(self, x):
        # Eq.3: T' = N( φ_MHSA(T) + T )
        x = self.norm1(x + self.drop1(self.attn(x)))
        # Eq.4: FFN with residual
        x = self.norm2(x + self.ffn(x))
        # Eq.5: SE recalibration
        return self.se(x)


class ViTSE(nn.Module):
    """
    ViT-SE Transformer (paper Section: Modified vision transformer).

    Eq.1 : Positional encoding T0 = T0 + Pn
           Pn[k] = {sin(ω·k), cos(ω·k)} at predefined frequency Pf

    Eq.6 : CLS-token classification head
           T_CLS = T[0]
           T_FC  = W·T_CLS + b
           C_k   = softmax(T_FC)

    Dimension scaling vs paper:
      Paper : 384×384, patch=16, embed=1024, depth=24, heads=16
      Here  : 64×64,   patch=8,  embed=256,  depth=6,  heads=4
    """
    def __init__(self, image_size=64, patch_size=8, in_channels=3,
                 num_classes=10, embed_dim=256, depth=6, num_heads=4,
                 mlp_ratio=4.0, dropout=0.1, se_reduction=16):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2
        patch_dim   = in_channels * patch_size * patch_size

        # Patch embedding (linear projection, paper: "patch embedding with
        # 16×16 patch sizes")
        self.patch_embed = nn.Sequential(
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        # CLS token (Eq.6: T_CLS = T[0])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Eq.1 : Positional encoding T0 = T0 + Pn
        # Using learnable positional encoding (standard ViT practice that
        # achieves Eq.1's purpose of retaining spatial information)
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # Transformer blocks with SE (Fig.3: 3 transformer+SE stacks)
        # Paper has 3 separate transformer column stacks in Fig.3;
        # depth=6 distributes these as 6 sequential blocks (faithful to
        # paper's "3 blocks" concept but as a single sequential stack)
        self.blocks = nn.ModuleList([
            TransformerSEBlock(embed_dim, num_heads, mlp_ratio,
                               dropout, se_reduction)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Eq.6 : T_FC = W·T_CLS + b  then softmax
        self.head = nn.Linear(embed_dim, num_classes)   # W·T_CLS + b in Eq.6

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _to_patches(self, x):
        """Reshape image to flat patch tokens."""
        B, C, H, W = x.shape; p = self.patch_size
        x = x.reshape(B, C, H//p, p, W//p, p)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        return x.reshape(B, (H//p)*(W//p), C*p*p)

    def forward(self, x):
        B = x.shape[0]
        tokens = self.patch_embed(self._to_patches(x))          # patch embed
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)                # prepend CLS
        # Eq.1 : T0 = T0 + Pn  (add positional encoding)
        tokens = self.pos_drop(tokens + self.pos_embed)
        for blk in self.blocks: tokens = blk(tokens)           # Eq.2–5
        tokens = self.norm(tokens)
        # Eq.6 : T_CLS = T[0], then FC + softmax
        t_cls  = tokens[:, 0]                                   # Eq.6: T_CLS
        return self.head(t_cls)                                  # Eq.6: T_FC

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — HyperParams & Training Utilities              ║
# ╚══════════════════════════════════════════════════════════╝

@dataclass
class HyperParams:
    # Paper Table 4 defaults
    lr:           float = 1e-4     # Table 4: α = 0.0001
    weight_decay: float = 1e-4
    dropout:      float = 0.2      # Table 4: dropout rate = 0.2
    embed_dim:    int   = 256
    depth:        int   = 6
    num_heads:    int   = 4
    mlp_ratio:    float = 4.0      # paper: FFN dim = 4 × embed_dim
    se_reduction: int   = 16
    batch_size:   int   = SAFE_BATCH


def build_model(hp: HyperParams) -> ViTSE:
    return ViTSE(
        embed_dim=hp.embed_dim, depth=hp.depth, num_heads=hp.num_heads,
        mlp_ratio=hp.mlp_ratio, dropout=hp.dropout,
        se_reduction=hp.se_reduction,
    ).to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = model(imgs); loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    return (total_loss / total, correct / total,
            np.concatenate(all_preds), np.concatenate(all_labels))


def compute_miou(preds, labels, n_classes=NUM_CLASSES):
    """Paper Table 7: per-class IoU = TP/(TP+FP+FN), then mean."""
    ious = []
    for c in range(n_classes):
        tp = ((preds == c) & (labels == c)).sum()
        fp = ((preds == c) & (labels != c)).sum()
        fn = ((preds != c) & (labels == c)).sum()
        denom = tp + fp + fn
        ious.append(float(tp / denom) if denom > 0 else 0.0)
    return float(np.mean(ious))


def compute_dice(preds, labels, n_classes=NUM_CLASSES):
    """Paper Table 7: Dice = 2TP/(2TP+FP+FN)."""
    dice = []
    for c in range(n_classes):
        tp = ((preds == c) & (labels == c)).sum()
        fp = ((preds == c) & (labels != c)).sum()
        fn = ((preds != c) & (labels == c)).sum()
        denom = 2 * tp + fp + fn
        dice.append(float(2 * tp / denom) if denom > 0 else 0.0)
    return float(np.mean(dice))


def compute_kappa(preds, labels):
    """
    Paper Eq.17: Ξ = (Po - Pe) / (1 - Pe)
    Po = (TP+TN)/(TP+TN+FP+FN)
    Pe = ((TP+FP)(TP+FN) + (FN+TN)(FP+TN)) / N²
    Implemented as sklearn cohen_kappa for multi-class extension.
    """
    from sklearn.metrics import cohen_kappa_score
    return cohen_kappa_score(labels, preds)


@dataclass
class ExperimentResult:
    name:             str
    tuning_method:    str
    best_hp:          dict
    train_acc_curve:  List[float] = field(default_factory=list)
    val_acc_curve:    List[float] = field(default_factory=list)
    train_loss_curve: List[float] = field(default_factory=list)
    val_loss_curve:   List[float] = field(default_factory=list)
    final_accuracy:   float = 0.0
    final_f1:         float = 0.0
    final_precision:  float = 0.0
    final_recall:     float = 0.0
    final_miou:       float = 0.0
    final_dice:       float = 0.0
    final_kappa:      float = 0.0
    training_time_s:  float = 0.0
    n_params:         int   = 0
    confusion:        Optional[np.ndarray] = None


def full_train(hp: HyperParams, epochs: int = FINAL_EPOCHS,
               verbose: bool = True) -> Tuple[ViTSE, ExperimentResult]:
    """
    Paper Table 4:
      - lr = 0.0001 (α)
      - split = 80:20
      - optimizer = MFA (we use AdamW as the gradient optimizer;
        MFA is the hyperparameter optimizer, not the gradient optimizer)
    """
    train_loader, val_loader = get_dataloaders(hp.batch_size)
    model     = build_model(hp)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=hp.lr,
                                  weight_decay=hp.weight_decay)
    # Paper trains for fixed epochs with no explicit LR schedule mentioned;
    # cosine decay is a faithful addition that avoids oscillation at end
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()   # paper does not mention label smoothing
    scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    result = ExperimentResult(name="", tuning_method="", best_hp=vars(hp))
    t0 = time.time()
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc             = train_one_epoch(model, train_loader,
                                                      optimizer, criterion,
                                                      scaler)
        va_loss, va_acc, preds, lbs = evaluate(model, val_loader, criterion)
        scheduler.step()

        result.train_acc_curve.append(tr_acc)
        result.val_acc_curve.append(va_acc)
        result.train_loss_curve.append(tr_loss)
        result.val_loss_curve.append(va_loss)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state   = deepcopy(model.state_dict())

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"train acc={tr_acc:.4f} loss={tr_loss:.4f} | "
                  f"val acc={va_acc:.4f} loss={va_loss:.4f}")

    model.load_state_dict(best_state)
    _, _, preds, lbs = evaluate(model, val_loader, criterion)

    result.training_time_s = time.time() - t0
    result.final_accuracy  = accuracy_score(lbs, preds)
    result.final_f1        = f1_score(lbs, preds, average="weighted")
    result.final_precision = precision_score(lbs, preds, average="weighted",
                                             zero_division=0)
    result.final_recall    = recall_score(lbs, preds, average="weighted",
                                          zero_division=0)
    result.final_miou      = compute_miou(preds, lbs)
    result.final_dice      = compute_dice(preds, lbs)
    result.final_kappa     = compute_kappa(preds, lbs)
    result.confusion       = confusion_matrix(lbs, preds)
    result.n_params        = model.count_parameters()
    return model, result


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Quick Fitness Evaluator (used by MFA+Optuna)  ║
# ╚══════════════════════════════════════════════════════════╝

def quick_fitness(hp: HyperParams, epochs: int = QUICK_EPOCHS) -> float:
    """Short training run → val accuracy as fitness proxy."""
    try:
        train_loader, val_loader = get_dataloaders(hp.batch_size)
        model     = build_model(hp)
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=hp.lr,
                                      weight_decay=hp.weight_decay)
        criterion = nn.CrossEntropyLoss()
        scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
        for _ in range(epochs):
            train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        _, val_acc, _, _ = evaluate(model, val_loader, criterion)
        del model; torch.cuda.empty_cache()
        return float(val_acc)
    except Exception as e:
        print(f"    [fitness error] {e}")
        return 0.0


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Modified Firefly Algorithm                    ║
# ║  Paper Equations 7–13, Table 5                          ║
# ╚══════════════════════════════════════════════════════════╝

class ModifiedFireflyAlgorithm:
    """
    Faithful implementation of MFA (Wassan et al., 2025).

    Standard FA equations used:
      Eq.7  : δ(d) = δ₀ / (1 + γd²)          light intensity
      Eq.8  : λ(d) = λ₀ / (1 + γd²)          attractiveness (rational)
      Eq.9  : d_ij = ‖r_i − r_j‖             Euclidean distance
      Eq.10 : standard FA movement (baseline)

    MFA modifications:
      Eq.11 : ω = ω_max − (ω_max−ω_min)×Iter/(1+maxIter)  inertia decay
      Eq.12 : ψ = (1−e^(−rand()×Iter/maxIter))/(1+e^(−rand()×Iter/maxIter))
                  NOTE: Eq.12 standalone uses rand(); Eq.13's actual movement
                  uses (rand()−0.5) for bidirectionality. Eq.13 is implemented
                  faithfully below.
      Eq.13 : r_i(n+1) = ω·r_i(n)
                        + λ₀·exp(−γd²)·(r_j−r_i)
                        + α·(1−e^(−(rand()−0.5)·Iter/maxIter))
                             /(1+e^(−(rand()−0.5)·Iter/maxIter))

    Table 5 parameters (where hardware permits):
      swarm size  = 50  → scaled to MFA_SWARM for Colab
      max_iter    = 700 → scaled to MFA_MAXITER for Colab
      γ           = 5        (faithful)
      λ₀          = 0.4      (faithful)
      α           = 0.01     (faithful)
      distance    = Euclidean (faithful, Eq.9)
    """

    # Hyperparameter search space (same 5 params as original code)
    BOUNDS = {
        "lr":           (1e-5, 1e-2),
        "weight_decay": (1e-5, 1e-2),
        "dropout":      (0.05, 0.30),
        "mlp_ratio":    (2.0,  6.0),
        "se_reduction": (4,    32),
    }
    HP_KEYS = list(BOUNDS.keys())
    DIM     = len(HP_KEYS)

    def __init__(self,
                 n_fireflies: int   = MFA_SWARM,
                 max_iter:    int   = MFA_MAXITER,
                 gamma:       float = 5.0,       # Table 5: γ = 5
                 lambda0:     float = 0.4,        # Table 5: λ₀ = 0.4
                 alpha:       float = 0.01,        # Table 5: α = 0.01
                 omega_max:   float = 0.9,         # paper mentions ω_max, ω_min
                 omega_min:   float = 0.2,
                 quick_epochs: int  = QUICK_EPOCHS):
        self.n        = n_fireflies
        self.T        = max_iter
        self.gamma    = gamma
        self.lam0     = lambda0
        self.alpha    = alpha
        self.w_max    = omega_max
        self.w_min    = omega_min
        self.qepochs  = quick_epochs

        # Population
        self.positions  = self._init_positions()
        self.brightness = np.full(n_fireflies, -np.inf)  # δ in paper
        self.best_pos   = None
        self.best_fit   = -np.inf
        self.fit_history = []

    # ── Population initialisation ───────────────────────────
    def _init_positions(self) -> np.ndarray:
        """Uniform random initialisation within bounds."""
        pos = np.zeros((self.n, self.DIM))
        for d, key in enumerate(self.HP_KEYS):
            lo, hi = self.BOUNDS[key]
            pos[:, d] = np.random.uniform(lo, hi, self.n)
        return pos

    def _pos_to_hp(self, pos: np.ndarray) -> HyperParams:
        hp = HyperParams()
        hp.lr           = float(np.clip(pos[0], *self.BOUNDS["lr"]))
        hp.weight_decay = float(np.clip(pos[1], *self.BOUNDS["weight_decay"]))
        hp.dropout      = float(np.clip(pos[2], *self.BOUNDS["dropout"]))
        hp.mlp_ratio    = float(np.clip(pos[3], *self.BOUNDS["mlp_ratio"]))
        hp.se_reduction = int(np.clip(round(pos[4]),
                                       *self.BOUNDS["se_reduction"]))
        hp.embed_dim  = 256; hp.depth = 6; hp.num_heads = 4
        hp.batch_size = SAFE_BATCH
        return hp

    # ── Paper equations ─────────────────────────────────────

    def _omega(self, t: int) -> float:
        """
        Eq.11 — Inertia factor (linearly decaying).
        ω = ω_max − (ω_max − ω_min) × Iter_number / (1 + max_Iter)
        """
        return self.w_max - (self.w_max - self.w_min) * t / (1 + self.T)

    def _psi(self, t: int) -> float:
        """
        Eq.13 — Geometric factor ψ (tan-sigmoid, bidirectional).
        Uses (rand()−0.5) as in the movement equation Eq.13.
        This allows negative values → bidirectional random walk.

        ψ = (1 − e^(−x)) / (1 + e^(−x))   where x=(rand()−0.5)×Iter/maxIter
        """
        r   = np.random.rand() - 0.5        # Eq.13: (rand() − 1/2)
        arg = r * t / max(self.T, 1)        # × Iter_number / max_Iter
        return (1 - np.exp(-arg)) / (1 + np.exp(-arg))

    def _distance(self, ri: np.ndarray, rj: np.ndarray) -> float:
        """
        Eq.9 — Euclidean (Cartesian) distance.
        d_ij = ‖r_i − r_j‖ = sqrt(Σ(r_ik − r_jk)²)
        """
        return float(np.sqrt(np.sum((ri - rj) ** 2)))

    def _attractiveness(self, d: float) -> float:
        """
        Eq.8 — Attractiveness (rational form, as defined in paper).
        λ(d) = λ₀ / (1 + γd²)

        NOTE: Eq.13's movement term uses λ₀·exp(−γd²) (exponential).
        The paper defines both forms. We use exponential in movement (Eq.13)
        and rational here for the standalone attractiveness definition (Eq.8).
        """
        return self.lam0 / (1 + self.gamma * d ** 2)

    def _attractiveness_exp(self, d: float) -> float:
        """
        Exponential attractiveness as used in Eq.13 movement equation:
        λ₀ · exp(−γ · d²)
        This matches Eq.10 (standard FA) and Eq.13 (MFA).
        """
        return self.lam0 * np.exp(-self.gamma * d ** 2)

    def _clip(self, pos: np.ndarray) -> np.ndarray:
        """Clip position to search space bounds."""
        for d, key in enumerate(self.HP_KEYS):
            lo, hi = self.BOUNDS[key]
            pos[d] = np.clip(pos[d], lo, hi)
        return pos

    # ── Main loop ────────────────────────────────────────────
    def run(self) -> HyperParams:
        print(f"\n[MFA] Initialising {self.n} fireflies  "
              f"(γ={self.gamma}, λ₀={self.lam0}, α={self.alpha})")
        print(f"      Iterations={self.T}, quick_epochs={self.qepochs}")

        # ── Step 1: Evaluate initial population ────────────
        # Paper pseudocode Step 1: "assign initial light intensity δ"
        for i in range(self.n):
            print(f"  Init firefly {i+1}/{self.n} ...", end="\r", flush=True)
            self.brightness[i] = quick_fitness(
                self._pos_to_hp(self.positions[i]), self.qepochs)
        print()

        best_idx      = int(np.argmax(self.brightness))
        self.best_pos = self.positions[best_idx].copy()
        self.best_fit = float(self.brightness[best_idx])
        self.fit_history.append(self.best_fit)
        print(f"  Init best fitness: {self.best_fit:.4f}  "
              f"(firefly {best_idx})")
        print(f"  Population fitness: "
              f"[{self.brightness.min():.4f} – {self.brightness.max():.4f}]")

        # ── Steps 3–4: Firefly movement & ranking ──────────
        for t in range(1, self.T + 1):
            omega = self._omega(t)                       # Eq.11
            moved = np.zeros(self.n, dtype=bool)

            # Paper pseudocode Step 3:
            # "For each firefly i, compare brightness δ_i with every other j.
            #  If δ_j > δ_i, move firefly i toward j."
            for i in range(self.n):
                for j in range(self.n):
                    if i == j:
                        continue
                    if self.brightness[j] > self.brightness[i]:
                        d   = self._distance(self.positions[i],
                                             self.positions[j])  # Eq.9
                        att = self._attractiveness_exp(d)         # Eq.13
                        psi = self._psi(t)                        # Eq.13

                        # Eq.13 — full MFA movement equation
                        self.positions[i] = (
                            omega * self.positions[i]             # ω·r_i(n)
                            + att * (self.positions[j]
                                     - self.positions[i])         # attraction
                            + self.alpha * psi                    # α·ψ
                        )
                        self._clip(self.positions[i])
                        moved[i] = True

            # Evaluate moved fireflies and update brightness
            for i in range(self.n):
                if moved[i]:
                    new_fit = quick_fitness(
                        self._pos_to_hp(self.positions[i]), self.qepochs)
                    # Elitist update: keep position if fitness improved
                    if new_fit > self.brightness[i]:
                        self.brightness[i] = new_fit

            # Paper Step 4: rank & retain best
            best_idx = int(np.argmax(self.brightness))
            if self.brightness[best_idx] > self.best_fit:
                self.best_fit = float(self.brightness[best_idx])
                self.best_pos = self.positions[best_idx].copy()
            self.fit_history.append(self.best_fit)

            n_moved = int(moved.sum())
            print(f"  MFA iter {t:3d}/{self.T} | "
                  f"ω={omega:.3f} | "
                  f"best={self.best_fit:.4f} | "
                  f"moved={n_moved}/{self.n} | "
                  f"pop=[{self.brightness.min():.3f}–"
                  f"{self.brightness.max():.3f}]")

        print(f"[MFA] Done. Best fitness: {self.best_fit:.4f}")
        best_hp = self._pos_to_hp(self.best_pos)
        print(f"      Best HP: lr={best_hp.lr:.2e}  "
              f"wd={best_hp.weight_decay:.2e}  "
              f"dropout={best_hp.dropout:.3f}  "
              f"mlp_ratio={best_hp.mlp_ratio:.2f}  "
              f"se_red={best_hp.se_reduction}")
        return best_hp


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Optuna (Bayesian TPE) Baseline               ║
# ╚══════════════════════════════════════════════════════════╝

def optuna_tune(n_trials: int = OPTUNA_TRIALS,
                quick_epochs: int = QUICK_EPOCHS) -> HyperParams:
    def objective(trial):
        hp = HyperParams(
            lr           = trial.suggest_float("lr",
                               1e-5, 1e-2, log=True),
            weight_decay = trial.suggest_float("weight_decay",
                               1e-5, 1e-2, log=True),
            dropout      = trial.suggest_float("dropout",      0.05, 0.30),
            mlp_ratio    = trial.suggest_float("mlp_ratio",    2.0,  6.0),
            se_reduction = trial.suggest_int("se_reduction",   4,    32),
            embed_dim=256, depth=6, num_heads=4, batch_size=SAFE_BATCH,
        )
        return quick_fitness(hp, quick_epochs)

    print(f"[Optuna] {n_trials} trials (Bayesian TPE) ...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    print(f"[Optuna] Best val_acc={study.best_value:.4f}  {best}")
    return HyperParams(
        lr=best["lr"], weight_decay=best["weight_decay"],
        dropout=best["dropout"], mlp_ratio=best["mlp_ratio"],
        se_reduction=best["se_reduction"],
        embed_dim=256, depth=6, num_heads=4, batch_size=SAFE_BATCH,
    )


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Paper-Default (Fixed) HPs  (Table 4)         ║
# ╚══════════════════════════════════════════════════════════╝

PAPER_HP = HyperParams(
    lr=1e-4,          # Table 4: α = 0.0001
    weight_decay=1e-4,
    dropout=0.2,      # Table 4: dropout rate = 0.2
    embed_dim=256,
    depth=6,
    num_heads=4,
    mlp_ratio=4.0,    # Table 4: feedforward dim = embed × 4
    se_reduction=16,
    batch_size=SAFE_BATCH,
)


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Monte Carlo Stability Analysis  (Eqs 14–17)  ║
# ╚══════════════════════════════════════════════════════════╝

def stability_analysis(best_result: ExperimentResult,
                       n_runs: int = 10) -> dict:
    """
    Paper Section: Ablation study 5 — stability via Monte Carlo simulation.
    200 independent runs in paper; scaled to n_runs for Colab.

    Statistical measures (paper Eqs 14–17):
      Eq.14: µ = (1/N) Σ z_i
      Eq.15: σ = sqrt((1/N) Σ (z_i−µ)²)
      Eq.16: κ (kurtosis) = [N(N+1)/((N−1)(N−2)(N−3))]·Σ((z_i−µ)/σ)⁴
                           − 3(N−1)²/((N−2)(N−3))
      Eq.17: Ξ (kappa coefficient) — computed on last run's preds/labels
    """
    print(f"\n[Monte Carlo] {n_runs} independent runs  "
          f"(paper uses 200; scaled for Colab)")
    hp   = HyperParams(**best_result.best_hp)
    accs = []

    for run in range(1, n_runs + 1):
        set_seed(SEED + run)
        _, r = full_train(hp, epochs=FINAL_EPOCHS, verbose=False)
        accs.append(r.final_accuracy * 100)
        print(f"  Run {run:3d}/{n_runs}: "
              f"acc={accs[-1]:.3f}%  "
              f"F1={r.final_f1:.4f}  "
              f"mIoU={r.final_miou:.4f}")

    z  = np.array(accs)
    N  = len(z)

    # Eq.14 — mean
    mu = float(z.mean())

    # Eq.15 — standard deviation
    sigma = float(np.sqrt(np.mean((z - mu) ** 2)))

    # Eq.16 — kurtosis (Fisher's excess kurtosis)
    if sigma > 0 and N > 3:
        kurtosis = (
            (N * (N + 1) / ((N - 1) * (N - 2) * (N - 3)))
            * np.sum(((z - mu) / sigma) ** 4)
            - 3 * (N - 1) ** 2 / ((N - 2) * (N - 3))
        )
    else:
        kurtosis = float("nan")

    stats = {
        "n_runs":   N,
        "min":      float(z.min()),
        "max":      float(z.max()),
        "mean_mu":  mu,
        "std_sigma": sigma,
        "kurtosis_kappa": float(kurtosis),
    }

    print(f"\n  ── Monte Carlo Statistics (Eqs 14–16) ──")
    print(f"  N (runs)          : {N}")
    print(f"  Min               : {stats['min']:.3f}%")
    print(f"  Max               : {stats['max']:.3f}%")
    print(f"  µ  (Eq.14)        : {stats['mean_mu']:.3f}%")
    print(f"  σ  (Eq.15)        : {stats['std_sigma']:.4f}%")
    print(f"  κ  (Eq.16)        : {stats['kurtosis_kappa']:.4f}")
    return stats


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Experiment Runner                            ║
# ╚══════════════════════════════════════════════════════════╝

def run_experiment(tuner: str) -> Tuple[ViTSE, ExperimentResult]:
    print(f"\n{'='*60}")
    print(f"  TUNER : {tuner.upper()}")
    print(f"  Epochs: {FINAL_EPOCHS}   Batch: {SAFE_BATCH}")
    print(f"{'='*60}")

    # Step 1: Hyperparameter optimisation
    if tuner == "fixed":
        best_hp = deepcopy(PAPER_HP)
        print("[Fixed] Using paper Table 4 hyperparameters")

    elif tuner == "mfa":
        mfa = ModifiedFireflyAlgorithm()   # uses module-level MFA_* constants
        best_hp = mfa.run()

    elif tuner == "optuna":
        best_hp = optuna_tune()

    # Step 2: Full training
    print(f"\n[Train] lr={best_hp.lr:.2e}  "
          f"wd={best_hp.weight_decay:.2e}  "
          f"dropout={best_hp.dropout:.2f}  "
          f"mlp_ratio={best_hp.mlp_ratio:.1f}  "
          f"se_red={best_hp.se_reduction}")
    model, result = full_train(best_hp, epochs=FINAL_EPOCHS, verbose=True)
    result.name          = tuner
    result.tuning_method = tuner

    # Step 3: Print results matching paper Table 7 format
    print(f"\n  ── Results ──────────────────────────────────")
    print(f"  Accuracy  (OA)  : {result.final_accuracy*100:.2f}%")
    print(f"  F1-score        : {result.final_f1:.4f}")
    print(f"  Precision       : {result.final_precision:.4f}")
    print(f"  Recall          : {result.final_recall:.4f}")
    print(f"  mIoU            : {result.final_miou:.4f}")
    print(f"  Dice            : {result.final_dice:.4f}")
    print(f"  Kappa (Ξ Eq.17) : {result.final_kappa:.4f}")
    print(f"  Time (s)        : {result.training_time_s:.0f}")
    print(f"  Params          : {result.n_params:,}")

    # Save model
    _save_model(model, result)
    return model, result


def _save_model(model: ViTSE, result: ExperimentResult):
    tag  = result.tuning_method
    wts  = os.path.join(OUTPUT_DIR, f"{tag}_model.pth")
    ckpt = os.path.join(OUTPUT_DIR, f"{tag}_checkpoint.pth")
    torch.save(model.state_dict(), wts)
    torch.save({
        "model_state_dict": model.state_dict(),
        "hyperparams":      result.best_hp,
        "final_accuracy":   result.final_accuracy,
        "final_f1":         result.final_f1,
        "final_miou":       result.final_miou,
        "final_kappa":      result.final_kappa,
        "training_time_s":  result.training_time_s,
        "n_params":         result.n_params,
        "image_size":       IMAGE_SIZE,
        "patch_size":       PATCH_SIZE,
        "num_classes":      NUM_CLASSES,
        "class_names":      CLASS_NAMES,
    }, ckpt)
    print(f"  Saved -> {wts}")
    print(f"  Saved -> {ckpt}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Visualisations                               ║
# ╚══════════════════════════════════════════════════════════╝

def plot_training_curves(results: List[ExperimentResult]):
    n    = len(results)
    fig, axes = plt.subplots(1, n, figsize=(n * 5, 4))
    if n == 1: axes = [axes]
    colors = ["#4C72B0", "#DD8452", "#55A868"]
    for i, r in enumerate(results):
        ax = axes[i]
        ep = range(1, len(r.val_acc_curve) + 1)
        ax.plot(ep, [v*100 for v in r.train_acc_curve],
                label="Train", lw=2, color=colors[i])
        ax.plot(ep, [v*100 for v in r.val_acc_curve],
                label="Val", lw=2, ls="--", color=colors[i], alpha=0.7)
        ax.set_title(f"{r.tuning_method.upper()}\n"
                     f"Best val={max(r.val_acc_curve)*100:.1f}%")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
        ax.legend(); ax.set_ylim(0, 102); ax.grid(alpha=0.3)
    plt.suptitle("Training Accuracy Curves (Fig.4 equivalent)", fontsize=12)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "training_curves.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


def plot_loss_curves(results: List[ExperimentResult]):
    n    = len(results)
    fig, axes = plt.subplots(1, n, figsize=(n * 5, 4))
    if n == 1: axes = [axes]
    colors = ["#4C72B0", "#DD8452", "#55A868"]
    for i, r in enumerate(results):
        ax = axes[i]
        ep = range(1, len(r.val_loss_curve) + 1)
        ax.plot(ep, r.train_loss_curve, label="Train Loss", lw=2, color=colors[i])
        ax.plot(ep, r.val_loss_curve,   label="Val Loss",   lw=2,
                ls="--", color=colors[i], alpha=0.7)
        ax.set_title(r.tuning_method.upper())
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Loss Curves (Fig.4 equivalent)", fontsize=12)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "loss_curves.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


def plot_confusion(result: ExperimentResult):
    cm_pct = (result.confusion.astype(float)
              / result.confusion.sum(axis=1, keepdims=True) * 100)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.4)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {result.tuning_method.upper()}\n"
                 f"Accuracy: {result.final_accuracy*100:.2f}%  "
                 f"Kappa Ξ: {result.final_kappa:.4f}")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f"confusion_{result.tuning_method}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


def plot_per_class_metrics(results: List[ExperimentResult]):
    """Paper Table 7: per-class precision, recall, F1, mIoU, Dice."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    x     = np.arange(NUM_CLASSES)
    bar_w = 0.8 / len(results)
    colors = ["#4C72B0", "#DD8452", "#55A868"]

    for idx, r in enumerate(results):
        cm  = r.confusion
        f1s, mious = [], []
        for c in range(NUM_CLASSES):
            tp = cm[c, c]
            fp = cm[:, c].sum() - tp
            fn = cm[c, :].sum() - tp
            denom_f1  = 2*tp + fp + fn
            denom_iou = tp + fp + fn
            f1s.append(2*tp/denom_f1   if denom_f1  > 0 else 0.0)
            mious.append(tp/denom_iou  if denom_iou > 0 else 0.0)
        offset = (idx - len(results)/2 + 0.5) * bar_w
        axes[0].bar(x + offset, f1s,   bar_w,
                    label=r.tuning_method.upper(),
                    color=colors[idx], alpha=0.85)
        axes[1].bar(x + offset, mious, bar_w,
                    label=r.tuning_method.upper(),
                    color=colors[idx], alpha=0.85)

    for ax, title in zip(axes, ["Per-Class F1 (Table 7)", "Per-Class mIoU (Table 7)"]):
        ax.set_xticks(x)
        ax.set_xticklabels(CLASS_NAMES, rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("Score"); ax.set_title(title)
        ax.legend(); ax.set_ylim(0, 1.1); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "per_class_metrics.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


def plot_tuner_comparison(results: List[ExperimentResult]):
    """Bar chart comparing Accuracy, F1, mIoU, Dice across tuners."""
    tuners  = [r.tuning_method.upper() for r in results]
    metrics = {
        "Accuracy (%)": [r.final_accuracy * 100 for r in results],
        "F1 (%)":       [r.final_f1 * 100       for r in results],
        "mIoU (%)":     [r.final_miou * 100      for r in results],
        "Dice (%)":     [r.final_dice * 100      for r in results],
    }
    x, w = np.arange(len(tuners)), 0.2
    fig, ax = plt.subplots(figsize=(11, 5))
    colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
    for idx, (label, vals) in enumerate(metrics.items()):
        bars = ax.bar(x + (idx - 1.5) * w, vals, w,
                      label=label, color=colors[idx])
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.3,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(tuners, fontsize=11)
    ax.set_ylabel("Score (%)"); ax.set_ylim(0, 115)
    ax.set_title("Tuner Comparison — Accuracy / F1 / mIoU / Dice")
    ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "tuner_comparison.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


def plot_mfa_convergence(mfa_result: ExperimentResult):
    """Plot MFA fitness history (Fig.7/9 equivalent)."""
    if not hasattr(mfa_result, "_fit_history"): return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(mfa_result._fit_history, marker="o", lw=2, color="#DD8452")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Best Fitness (val acc)")
    ax.set_title("MFA Convergence (Fig.7/9 equivalent)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "mfa_convergence.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved -> {p}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 12 — Summary Table & JSON Export                  ║
# ╚══════════════════════════════════════════════════════════╝

def print_summary(results: List[ExperimentResult]):
    """Print Table 7-style summary matching paper's metric columns."""
    print("\n" + "="*100)
    print(f"  RESULTS  (Table 7 equivalent)   Epochs={FINAL_EPOCHS}")
    print("="*100)
    hdr = (f"{'Tuner':<12} {'Accuracy':>10} {'F1':>8} "
           f"{'Precision':>10} {'Recall':>8} {'mIoU':>8} "
           f"{'Dice':>8} {'Kappa-Ξ':>9} {'Time(s)':>8}")
    print(hdr); print("-"*100)
    for r in sorted(results, key=lambda x: -x.final_accuracy):
        print(f"{r.tuning_method:<12} "
              f"{r.final_accuracy*100:>9.2f}% "
              f"{r.final_f1:>8.4f} "
              f"{r.final_precision:>10.4f} "
              f"{r.final_recall:>8.4f} "
              f"{r.final_miou:>8.4f} "
              f"{r.final_dice:>8.4f} "
              f"{r.final_kappa:>9.4f} "
              f"{r.training_time_s:>8.0f}")
    print("="*100)
    best = max(results, key=lambda r: r.final_accuracy)
    print(f"\n  WINNER : {best.tuning_method.upper()}  "
          f"({best.final_accuracy*100:.2f}% accuracy, "
          f"Kappa={best.final_kappa:.4f})")


def save_results_json(results: List[ExperimentResult]):
    path = os.path.join(OUTPUT_DIR, "results.json")
    data = []
    for r in results:
        d = {k: v for k, v in vars(r).items() if k != "confusion"}
        if r.confusion is not None:
            d["confusion"] = r.confusion.tolist()
        data.append(d)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Results JSON -> {path}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 13 — MAIN                                         ║
# ╚══════════════════════════════════════════════════════════╝

# ── Sanity check ───────────────────────────────────────────
_m = ViTSE().to(DEVICE)
with torch.no_grad():
    _o = _m(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE))
print(f"\nModel OK: output={_o.shape}  params={_m.count_parameters():,}")
del _m; torch.cuda.empty_cache()

# ── Load dataset once ──────────────────────────────────────
print("\nLoading dataset ...")
get_base_dataset()

# ── Run all three tuners ───────────────────────────────────
all_results: List[ExperimentResult] = []
all_models  = {}
mfa_instance = None   # keep reference for convergence plot

for tuner in TUNING_METHODS:
    set_seed()
    try:
        model, result = run_experiment(tuner)
        all_results.append(result)
        all_models[tuner] = model
    except Exception as e:
        import traceback
        print(f"\n!! {tuner} failed: {e}")
        traceback.print_exc()

print("\n\nAll experiments complete!")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 14 — Results, Plots & Monte Carlo                 ║
# ╚══════════════════════════════════════════════════════════╝

print_summary(all_results)
save_results_json(all_results)

plot_training_curves(all_results)
plot_loss_curves(all_results)
plot_tuner_comparison(all_results)
plot_per_class_metrics(all_results)
for r in all_results:
    plot_confusion(r)

# ── Monte Carlo stability (paper Ablation study 5) ─────────
# Set n_runs=200 in FULL mode to match paper; 10 for QUICK
MC_RUNS = 200 if RUN_MODE == "FULL" else 10
best_result = max(all_results, key=lambda r: r.final_accuracy)
mc_stats    = stability_analysis(best_result, n_runs=MC_RUNS)


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 15 — Final Report                                 ║
# ╚══════════════════════════════════════════════════════════╝

print("\n" + "="*65)
print("            FINAL EXPERIMENT REPORT")
print("="*65)
print(f"Dataset      : EuroSAT (27,000 samples, 10 classes, 64×64)")
print(f"Model        : ViT-SE (embed={PAPER_HP.embed_dim}, "
      f"depth={PAPER_HP.depth}, heads={PAPER_HP.num_heads})")
print(f"Patch size   : {PATCH_SIZE}×{PATCH_SIZE} → "
      f"{(IMAGE_SIZE//PATCH_SIZE)**2} patches per image")
print(f"Final epochs : {FINAL_EPOCHS}")
print(f"Device       : {DEVICE}")
print(f"Mode         : {RUN_MODE}")
print()

sorted_res = sorted(all_results, key=lambda r: -r.final_accuracy)
print("RANKING (Table 10 equivalent):")
for i, r in enumerate(sorted_res, 1):
    print(f"  {i}. {r.tuning_method.upper():<10}  "
          f"Acc={r.final_accuracy*100:.2f}%  "
          f"F1={r.final_f1:.4f}  "
          f"mIoU={r.final_miou:.4f}  "
          f"Dice={r.final_dice:.4f}  "
          f"Kappa={r.final_kappa:.4f}  "
          f"Time={r.training_time_s:.0f}s")

print()
print("MONTE CARLO STABILITY (Table 9 equivalent):")
print(f"  Runs (N)  : {mc_stats['n_runs']}  "
      f"(paper: 200)")
print(f"  Min       : {mc_stats['min']:.3f}%")
print(f"  Max       : {mc_stats['max']:.3f}%")
print(f"  µ  Eq.14  : {mc_stats['mean_mu']:.3f}%")
print(f"  σ  Eq.15  : {mc_stats['std_sigma']:.4f}%")
print(f"  κ  Eq.16  : {mc_stats['kurtosis_kappa']:.4f}")
print()
print(f"BEST TUNER : {sorted_res[0].tuning_method.upper()}")
print(f"BEST ACC   : {sorted_res[0].final_accuracy*100:.2f}%")
print()
print("FAITHFULNESS NOTES:")
print("  Eq.11 ω   :  exact")
print("  Eq.12/13 ψ:  (rand()-0.5) bidirectional")
print("  Eq.13 mvmt:  exponential attractiveness λ₀·exp(-γd²)")
print("  Eq.5  SE  :  global-avg-pool → FC → sigmoid")
print("  Eq.6  CLS :  T[0] → FC → softmax")
print("  Eq.14-16  :  µ, σ, κ exact formulas")
print("  Dimensions:  scaled (64×64 vs 384×384, embed 256 vs 1024)")
print("="*65)